In [1]:
import os
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

import numpy as np
from collections import Counter
import rasterio
from rasterio.features import rasterize
from rasterio import windows
import geopandas as gpd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import SegformerForSemanticSegmentation
from tqdm import tqdm
import segmentation_models_pytorch as smp
import re
from pathlib import Path
from collections import defaultdict
from functools import lru_cache
import cv2
import albumentations as A

/home/alex/projects/NewPipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================ НАСТРОЙКИ ============================

# ROOT_TIF = Path('/data/mlsystem2/prepared_images/kanopus/')
# ROOT_GEO = Path('/home/alex/projects/NewPipeline/originals/hlam_mask_main/')

# ROOT_TIF = Path('/data/mlsystem2/prepared_images/kanopus/')
# ROOT_GEO = Path('/home/alex/projects/NewPipeline/originals/hlam_mask_main_without_test/')

TARGET_REGIONS = ['irkutsk2', 'irkutsk', 'hlam3', 'hlam2', 'Olskij']
HN_TOKEN = 'hn'
DUP_SUFFIX = re.compile(r'\s*\(\d+\)$')    

PATCH_SIZE = 512
STRIDE = 256
BATCH_SIZE = 16
NUM_EPOCHS = 20
LEARNING_RATE = 0.0001
HARD_NEG_WEIGHT = 8.0   # вес hard-negative патчей в сэмплере
sample_weights_val = 7
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используется устройство: {DEVICE}')

ROOT_TIF = Path('/data/mlsystem2/prepared_images/kanopus')
ROOT_GEO = Path('/home/alex/projects/NewPipeline/originals/mask_hlam_origin_without_test')


HN_TOKEN = 'hn'
DUP_SUFFIX = re.compile(r'\s*\(\d+\)$')

ROLE_COL = '_mlsystem2_role'
POSITIVE_VALUES = {'positive', 'pos', '1', 'target'}
HN_VALUES = {'hard_negative', 'hard-negative', 'hardnegative', 'hn', 'hneg', 'neg'}

Используется устройство: cuda


In [3]:
def build_train_transform():
    """
    Аугментации для train. Размер патча сохраняется; маска трансформируется
    согласованно с изображением. Работает с 4-канальным RGBN.
    """
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(
            scale=(0.9, 1.1),                    # ±10% масштаба
            translate_px=(-32, 32),              # ±32 px (≈6% для патча 512)
            rotate=(-15, 15),                    # ±15° поворот
            border_mode=cv2.BORDER_REFLECT_101,  # зеркальная заливка, без чёрных краёв
            p=0.4,
        ),
        # --- Шум/размытие: имитируют разные атмосферные условия ---
        A.OneOf([
            A.GaussNoise(var_limit=(5.0, 30.0), p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        ], p=0.3),
        # --- Яркость/контраст: одинаково для всех каналов, не ломает спектр ---
        A.RandomBrightnessContrast(
            brightness_limit=0.15, contrast_limit=0.15,
            per_channel=False, p=0.3,
        ),
    ])

In [4]:
# ============================ ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ============================
def _scan_regions(root: Path, allowed=None):
    if not root.is_dir():
        raise FileNotFoundError(f'Нет каталога со снимками: {root}')
    regions = [p.name for p in root.iterdir() if p.is_dir()]
    if allowed is not None:
        regions = [r for r in regions if r in allowed]
    regions.sort(key=len, reverse=True)      # 'wave_2_Upload_01' раньше 'wave'
    return regions


def _parse_mask(stem: str, regions):
    """
    'hlam3_KVI_22911_...'              -> ('hlam3',  'KVI_22911_...')
    'irkutsk_KV3_30937_...'            -> ('irkutsk','KV3_30937_...')
    'irkutsk_hn_KV3_...' (legacy)      -> ('irkutsk','KV3_...')   # hn_ префикс опционален
    """
    for region in regions:
        prefix = region + '_'
        if not stem.startswith(prefix):
            continue
        rest = stem[len(prefix):]
        if rest.startswith(HN_TOKEN + '_'):
            rest = rest[len(HN_TOKEN) + 1:]
        return region, rest
    return None

def normalize_image(img):
    img = img.astype(np.float32)
    for c in range(img.shape[0]):
        min_val = img[c].min()
        max_val = img[c].max()
        if max_val - min_val > 1e-6:
            img[c] = (img[c] - min_val) / (max_val - min_val)
        else:
            img[c] = 0
    return img
    
# ============================ ОСНОВНАЯ ФУНКЦИЯ ============================
def get_file_pairs(root_tif=ROOT_TIF, root_geo=ROOT_GEO,
                   allowed_regions=TARGET_REGIONS, verbose=True):
    """
    Возвращает список пар (tif_path, geojson_path).

    Один GeoJSON содержит и positive, и hard_negative; разделение — по колонке
    _mlsystem2_role внутри файла, поэтому target и hn указывают на один путь.
    """
    regions = _scan_regions(root_tif, allowed=allowed_regions)
    if verbose:
        print(f'Регионы ({len(regions)}): {regions}')

    # --- 1. Индекс tif: {(region, base): Path} -------------------------------
    tif_index = {}
    for region in regions:
        for tif in (root_tif / region).rglob('*'):
            if not tif.is_file() or tif.suffix.lower() not in ('.tif', '.tiff'):
                continue
            tif_index.setdefault((region, DUP_SUFFIX.sub('', tif.stem)), tif)
    if verbose:
        print(f'Проиндексировано tif: {len(tif_index)}')

    # --- 2. Индекс geojson: {(region, base): [Path, ...]} --------------------
    geo_index = defaultdict(list)
    for geo in sorted(root_geo.glob('*.geojson')):
        parsed = _parse_mask(geo.stem, regions)
        if parsed is None:
            if verbose:
                print(f'[регион не опознан] {geo.name}')
            continue
        region, base = parsed
        base = DUP_SUFFIX.sub('', base)
        geo_index[(region, base)].append(geo)

    # --- 3. Сводим в пары ----------------------------------------------------
    pairs, missing_tif = [], []
    for (region, base), geos in sorted(geo_index.items()):
        tif = tif_index.get((region, base))
        if tif is None:
            missing_tif.append(f'{region}/{base}')
            continue
        # если нашлось несколько geojson на один base — предупреждаем и берём все
        # (каждый файл = отдельный набор ролей; редкий случай, но пусть будет)
        for geo in geos:
            pairs.append((str(tif), str(geo)))

    if verbose and missing_tif:
        print(f'Нет tif для {len(missing_tif)} масок. Примеры (первые 10):')
        for m in missing_tif[:10]:
            print('  ', m)
        for region in {m.split('/')[0] for m in missing_tif[:3]}:
            sample = [k[1] for k in tif_index if k[0] == region][:3]
            print(f'   пример base в {region}/: {sample}')

    return pairs


# ============================ ЗАГРУЗКА МАСОК ============================
@lru_cache(maxsize=256)
def _read_geojson_cached(geo_path_str: str) -> gpd.GeoDataFrame:
    """Читает geojson один раз и нормализует колонку роли."""
    gdf = gpd.read_file(geo_path_str)
    if gdf.empty:
        return gdf
    if ROLE_COL in gdf.columns:
        gdf = gdf.copy()
        gdf['_role_norm'] = (gdf[ROLE_COL].fillna('positive')
                             .astype(str).str.lower().str.strip())
    else:
        gdf = gdf.copy()
        gdf['_role_norm'] = 'positive'
    return gdf


def geojson_to_mask_for_window(geojson_path, transform, out_shape,
                               role='positive'):
    """
    Растрирует в окне только полигоны указанной роли.
    role='positive'       — все не-hn полигоны
    role='hard_negative'  — только hn-полигоны
    """
    gdf = _read_geojson_cached(str(geojson_path))
    if gdf.empty:
        return np.zeros(out_shape, dtype=np.uint8)

    if role == 'hard_negative':
        sub = gdf[gdf['_role_norm'].isin(HN_VALUES)]
    else:
        sub = gdf[gdf['_role_norm'].isin(POSITIVE_VALUES)]

    if sub.empty:
        return np.zeros(out_shape, dtype=np.uint8)

    shapes = [(geom, 1) for geom in sub.geometry]
    return rasterize(shapes, out_shape=out_shape, transform=transform,
                     fill=0, dtype='uint8')


# ============================ ГЕНЕРАЦИЯ КООРДИНАТ ============================
def build_all_patch_coords(pairs, patch_size, stride, context=0):
    coords = []
    for tif_path, geo_path in pairs:
        with rasterio.open(tif_path) as src:
            width, height = src.width, src.height
        for y in range(context, height - patch_size - context + 1, stride):
            for x in range(context, width - patch_size - context + 1, stride):
                coords.append((tif_path, geo_path, x, y))
    return coords


# ============================ ДАТАСЕТ ============================
class OnTheFlyDataset(Dataset):
    def __init__(self, patch_coords, patch_size, transform=None,
                 return_positive_info=False, return_hn_info=False):
        self.patch_coords = patch_coords
        self.patch_size = patch_size
        self.transform = transform              # albumentations.Compose или None
        self.return_positive_info = return_positive_info
        self.return_hn_info = return_hn_info

    def __len__(self):
        return len(self.patch_coords)

    def __getitem__(self, idx):
        tif_path, geo_path, x, y = self.patch_coords[idx]

        with rasterio.open(tif_path) as src:
            window = windows.Window(x, y, self.patch_size, self.patch_size)
            img = src.read(window=window)               # (C, H, W) uint8
            win_transform = src.window_transform(window)

        positive_mask = geojson_to_mask_for_window(
            geo_path, win_transform, (self.patch_size, self.patch_size),
            role='positive')
        hn_mask = geojson_to_mask_for_window(
            geo_path, win_transform, (self.patch_size, self.patch_size),
            role='hard_negative')

        # hn_ratio считаем по «сырой» маске ДО аугментации — этот индикатор
        # используется только на этапе сканирования (find_hard_negative_indices),
        # где трансформа не применяется.
        hn_ratio = float(hn_mask.mean())

        mask = positive_mask.copy()
        mask[hn_mask == 1] = 0

        # --- АУГМЕНТАЦИЯ -------------------------------------------------
        if self.transform is not None:
            # (C, H, W) -> (H, W, C) для albumentations
            img_hwc = np.transpose(img, (1, 2, 0))
            out = self.transform(image=img_hwc, mask=mask)
            img = np.transpose(out['image'], (2, 0, 1))   # обратно в (C, H, W)
            mask = out['mask']
            if mask.dtype != np.uint8:
                mask = mask.astype(np.uint8)
        # -----------------------------------------------------------------

        img = normalize_image(img)                        # (C, H, W) float32 in [0,1]
        img = torch.tensor(img, dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.long)

        pos_ratio = (mask == 1).float().mean().item()

        if self.return_positive_info and self.return_hn_info:
            return img, mask, pos_ratio, hn_ratio
        elif self.return_positive_info:
            return img, mask, pos_ratio
        elif self.return_hn_info:
            return img, mask, hn_ratio
        return img, mask

        
# ============================ ВЕСА И СЭМПЛЕР ============================
def compute_class_weights_and_sampler(dataset, hard_neg_indices=None,
                                      hard_neg_weight=HARD_NEG_WEIGHT,
                                      sample_weights_val=7.0):
    positive_ratios = []
    labels_present = []
    print("Сканирование train-патчей для вычисления статистики...")
    for i in tqdm(range(len(dataset))):
        _, mask, pos_ratio = dataset[i]         # [fix] dataset[I] -> dataset[i]
        positive_ratios.append(pos_ratio)
        labels_present.append(1 if pos_ratio > 0.001 else 0)

    positive_ratios = np.array(positive_ratios)
    labels_present = np.array(labels_present)

    sum_pos = positive_ratios.sum()
    sum_neg = (1 - positive_ratios).sum()
    weight_pos = 1.0 / (sum_pos + 1e-6)
    weight_neg = 1.0 / (sum_neg + 1e-6)
    mean_weight = (weight_pos + weight_neg) / 2
    weight_pos /= mean_weight
    weight_neg /= mean_weight
    class_weights = torch.tensor([weight_neg, weight_pos], dtype=torch.float32)

    sample_weights = np.where(labels_present == 1,
                              sample_weights_val, 1.0).astype(np.float64)
    if hard_neg_indices is not None:
        for idx in hard_neg_indices:
            sample_weights[idx] = max(sample_weights[idx], hard_neg_weight)

    sample_weights = sample_weights / sample_weights.sum()
    sampler = WeightedRandomSampler(sample_weights,
                                    num_samples=len(dataset),
                                    replacement=True)
    return class_weights, sampler


def find_hard_negative_indices(dataset):
    hn_indices = []
    print("Поиск hard-negative патчей...")
    for i in tqdm(range(len(dataset))):
        _, _, hn_ratio = dataset[i]
        if hn_ratio > 0.0:
            hn_indices.append(i)
    return hn_indices

In [5]:
# ============================ ПОДГОТОВКА ДАННЫХ ============================
pairs = get_file_pairs()
print(f'\nНайдено пар файлов: {len(pairs)}')
# Теперь одна пара на geojson, но внутри могут быть оба класса:
# посмотрим, сколько файлов реально содержат hn, а сколько — только positive.
from collections import Counter
role_stats = Counter()
for _, geo_path in pairs:
    g = _read_geojson_cached(str(geo_path))
    if g.empty:
        role_stats['empty'] += 1
        continue
    has_hn  = bool(g['_role_norm'].isin(HN_VALUES).any())
    has_pos = bool(g['_role_norm'].isin(POSITIVE_VALUES).any())
    if has_pos and has_hn:  role_stats['both'] += 1
    elif has_pos:           role_stats['positive_only'] += 1
    elif has_hn:            role_stats['hn_only'] += 1
    else:                   role_stats['unknown'] += 1
print(f'  по ролям: {dict(role_stats)}')

PICS_DIR = str(ROOT_TIF)
MASK_DIR = str(ROOT_GEO)

# патчи
all_coords = build_all_patch_coords(pairs, PATCH_SIZE, STRIDE)
print(f'Всего возможных патчей: {len(all_coords)}')

train_coords, temp_coords = train_test_split(all_coords, test_size=0.4, random_state=42)
val_coords,   test_coords = train_test_split(temp_coords, test_size=0.5, random_state=42)
print(f'Train: {len(train_coords)}, Val: {len(val_coords)}, Test: {len(test_coords)}')

# скан hard-negative в train
train_dataset_info = OnTheFlyDataset(train_coords, PATCH_SIZE, return_positive_info=True)
train_dataset_hn   = OnTheFlyDataset(train_coords, PATCH_SIZE, return_hn_info=True)
hard_neg_indices = find_hard_negative_indices(train_dataset_hn)
print(f'Hard-negative патчей в train: {len(hard_neg_indices)} из {len(train_coords)}')



Регионы (5): ['irkutsk2', 'irkutsk', 'Olskij', 'hlam3', 'hlam2']
Проиндексировано tif: 514
[регион не опознан] irkutks_KV5_24818_25736-01_KANOPUS_20230618_035304_20.L2.PMS.SCN05.geojson

Найдено пар файлов: 46
  по ролям: {'positive_only': 32, 'both': 11, 'hn_only': 3}
Всего возможных патчей: 18754
Train: 11252, Val: 3751, Test: 3751
Поиск hard-negative патчей...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11252/11252 [01:02<00:00, 179.22it/s]

Hard-negative патчей в train: 517 из 11252


In [6]:
class_weights, train_sampler = compute_class_weights_and_sampler(
    train_dataset_info,
    hard_neg_indices=hard_neg_indices,
    hard_neg_weight=HARD_NEG_WEIGHT,
    sample_weights_val=sample_weights_val,
)

train_transform = build_train_transform()

train_dataset = OnTheFlyDataset(train_coords, PATCH_SIZE, transform=train_transform)
val_dataset   = OnTheFlyDataset(val_coords,   PATCH_SIZE, transform=None)
test_dataset  = OnTheFlyDataset(test_coords,  PATCH_SIZE, transform=None)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


Сканирование train-патчей для вычисления статистики...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11252/11252 [00:59<00:00, 189.50it/s]
/tmp/ipykernel_17113/1176453918.py:19: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 30.0), p=1.0),
/tmp/ipykernel_17113/1176453918.py:23: UserWarning: Argument(s) 'per_channel' are not valid for transform RandomBrightnessContrast
  A.RandomBrightnessContrast(


In [7]:


# ============================ МОДЕЛЬ ============================
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    num_labels=2,
    ignore_mismatched_sizes=True,
    use_safetensors=True
)
first_conv = None
for module in model.modules():
    if isinstance(module, nn.Conv2d) and module.in_channels == 3:
        first_conv = module
        break
if first_conv is None:
    raise ValueError("Не найден свёрточный слой с 3 входными каналами")
new_conv = nn.Conv2d(4, first_conv.out_channels,
                     kernel_size=first_conv.kernel_size,
                     stride=first_conv.stride, padding=first_conv.padding,
                     bias=first_conv.bias is not None)
with torch.no_grad():
    new_conv.weight[:, :3] = first_conv.weight
    new_conv.weight[:, 3] = first_conv.weight[:, 0]
for name, module in model.named_modules():
    if module is first_conv:
        parent_name = name.rsplit('.', 1)[0] if '.' in name else ''
        parent = model.get_submodule(parent_name) if parent_name else model
        attr_name = name.rsplit('.', 1)[-1] if '.' in name else name
        setattr(parent, attr_name, new_conv)
        print(f"Заменён слой: {name}")
        break
model.config.num_channels = 4
model.to(DEVICE)
print(f'Модель загружена: {model.config.num_labels} классов, {model.config.num_channels} каналов')

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b0-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([2]) in the model instantiated
- decode_head.classifier.weight: found shape torch.Size([150, 256, 1, 1]) in the checkpoint and torch.Size([2, 256, 1, 1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Заменён слой: segformer.encoder.patch_embeddings.0.proj
Модель загружена: 2 классов, 4 каналов


In [8]:
# ============================ LOSS, OPTIMIZER ============================
tversky_loss = smp.losses.TverskyLoss(mode='multiclass', alpha=0.75, beta=0.25, from_logits=True)
ce_loss = torch.nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))

def combined_loss(logits, targets):
    return 0.25 * ce_loss(logits, targets) + 0.75 * tversky_loss(logits, targets)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

In [9]:
# ============================ ОБУЧЕНИЕ ============================
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [train]')
    for images, masks in pbar:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        logits = outputs.logits
        logits = torch.nn.functional.interpolate(logits, size=masks.shape[-2:],
                                                mode='bilinear', align_corners=False)
        loss = combined_loss(logits, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        pbar.set_postfix({'loss': loss.item()})
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs = model(images)
            logits = outputs.logits
            logits = torch.nn.functional.interpolate(logits, size=masks.shape[-2:],
                                                    mode='bilinear', align_corners=False)
            loss = combined_loss(logits, masks)
            val_loss += loss.item() * images.size(0)
    val_loss /= len(val_loader.dataset)
    print(f'Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}')
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model_segformer_hlam_main_3_without_test.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 10:
            print('Ранняя остановка.')
            break

Epoch 1/20 [train]: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 704/704 [02:26<00:00,  4.81it/s, loss=0.104]


Epoch 1: Train Loss = 0.4146, Val Loss = 0.2122


Epoch 2/20 [train]: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 704/704 [02:25<00:00,  4.83it/s, loss=0.0602]


Epoch 2: Train Loss = 0.3326, Val Loss = 0.1375


Epoch 3/20 [train]: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 704/704 [02:24<00:00,  4.86it/s, loss=0.0233]


...

Epoch 18: Train Loss = 0.1415, Val Loss = 0.0930


Epoch 19/20 [train]: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 704/704 [02:25<00:00,  4.82it/s, loss=0.001]


Epoch 19: Train Loss = 0.1390, Val Loss = 0.0847


Epoch 20/20 [train]: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 704/704 [02:25<00:00,  4.83it/s, loss=0.0731]


Epoch 20: Train Loss = 0.1338, Val Loss = 0.0895


In [10]:
# ============================ ОЦЕНКА ============================
model.load_state_dict(torch.load('best_model_segformer_hlam_main_3_without_test.pth'))
model.eval()

all_preds = []
all_targets = []
with torch.no_grad():
    for images, masks in tqdm(test_loader, desc='Оценка на тесте'):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        outputs = model(images)
        logits = outputs.logits
        logits = torch.nn.functional.interpolate(logits, size=masks.shape[-2:],
                                                mode='bilinear', align_corners=False)
        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.cpu().numpy().flatten())
        all_targets.append(masks.cpu().numpy().flatten())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

print('\nОтчёт по классам на тестовом наборе:')
print(classification_report(all_targets, all_preds, target_names=['фон', 'захламление']))

Оценка на тесте: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 235/235 [00:31<00:00,  7.54it/s]



Отчёт по классам на тестовом наборе:
              precision    recall  f1-score   support

         фон       1.00      1.00      1.00 981180149
 захламление       0.32      0.94      0.48   2121995

    accuracy                           1.00 983302144
   macro avg       0.66      0.97      0.74 983302144
weighted avg       1.00      1.00      1.00 983302144



In [ ]:
# alpha=0.8, beta=0.2,
# best_model_segformer_hlam_main_5

In [ ]:
# best_model_segformer_hlam_main_4
# Epoch 17: Train Loss = 0.2038, Val Loss = 0.0793
 # 0.25 * ce_loss(logits, targets) + 0.75
# alpha=0.75, beta=0.25,
# Отчёт по классам на тестовом наборе:
#               precision    recall  f1-score   support

#          фон       1.00      1.00      1.00 1071731740
#  захламление       0.41      0.94      0.57   2796516

#     accuracy                           1.00 1074528256
#    macro avg       0.71      0.97      0.79 1074528256
# weighted avg       1.00      1.00      1.00 1074528256

In [ ]:
# Epoch 8: Train Loss = 0.1042, Val Loss = 0.0884
# tversky_loss = smp.losses.TverskyLoss(mode='multiclass', alpha=0.6, beta=0.4, from_logits=True)
# return 0.3 * ce_loss(logits, targets) + 0.7 * tversky_loss(logits, targets)
# Отчёт по классам на тестовом наборе:
#               precision    recall  f1-score   support

#          фон       1.00      1.00      1.00 755554639
#  захламление       0.62      0.92      0.74   2827953

#     accuracy                           1.00 758382592
#    macro avg       0.81      0.96      0.87 758382592
# weighted avg       1.00      1.00      1.00 758382592

In [ ]:
# Epoch 6: Train Loss = 0.1143, Val Loss = 0.0963
# tversky_loss = smp.losses.TverskyLoss(mode='multiclass', alpha=0.6, beta=0.4, from_logits=True)
# return 0.25 * ce_loss(logits, targets) + 0.75 * tversky_loss(logits, targets)
# Отчёт по классам на тестовом наборе:
#               precision    recall  f1-score   support

#          фон       1.00      1.00      1.00 1071731740
#  захламление       0.73      0.88      0.80   2796516

#     accuracy                           1.00 1074528256
#    macro avg       0.86      0.94      0.90 1074528256
# weighted avg       1.00      1.00      1.00 1074528256


<!-- 0.6 alpha 0.4 beta
0.4 ce
0.6 tver
Отчёт по классам на тестовом наборе:
              precision    recall  f1-score   support

         фон       1.00      1.00      1.00 710552638
 захламление       0.72      0.91      0.80    906178

    accuracy                           1.00 711458816
   macro avg       0.86      0.96      0.90 711458816
weighted avg       1.00      1.00      1.00 711458816 -->

<!-- 0.6 alpha 0.4 beta
0.5 ce
0.5 tver
Оценка на тесте: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 170/170 [00:23<00:00,  7.35it/s]

Отчёт по классам на тестовом наборе:
              precision    recall  f1-score   support

         фон       1.00      1.00      1.00 710552638
 захламление       0.30      0.95      0.45    906178

    accuracy                           1.00 711458816
   macro avg       0.65      0.98      0.73 711458816
weighted avg       1.00      1.00      1.00 711458816 -->